# Featureの追加

---

OpenHPC環境に新しいFeature（計算ノードグループ）を追加します。FeatureからNodesetが自動生成され、既存または新規のPartitionに割り当てられます。

## 概要

このNotebookでは、既存のOpenHPC環境に対して以下の操作を行います：

1. 新しいFeatureの定義（計算ノードのハードウェア特性とVCユニット）
2. NodesetをDynamic NodesによりFeatureから自動生成（`ns-{feature_name}`）
3. Partition設定：既存のPartitionへの追加、または新規Partitionの作成
4. Slurm設定の更新（slurm.conf/hosts.ohpc）
5. mdx VMの起動（mdx環境の場合のみ）
6. VCノードの起動

### 追加パターン

**パターン1: 既存のPartitionに新しいFeatureを追加**
- 例: `gpu-partition` に Azure の A100 ノード（Feature: `gpu-a100-azure`）を追加
- Nodeset `ns-gpu-a100-azure` が自動生成され、既存の `gpu-partition` に追加される

**パターン2: 新しいPartitionを作成**
- 例: 新しい `highmem` Partitionを作成
- Feature `highmem-512gb-aws` を定義すると、Nodeset `ns-highmem-512gb-aws` が自動生成され、新規Partition `highmem` に割り当てられる

### 前提条件

- 初期構築（010, 020）が完了していること
- VCCアクセストークンが有効であること
- 既存環境のUnitGroup名を把握していること

## 準備

既存のOpenHPC環境にアクセスするための準備を行います。

### VCCアクセストークンの入力

VCノードを操作するためのアクセストークンを入力してください。

In [ ]:
from getpass import getpass
vcc_access_token = getpass()

入力されたアクセストークンが正しいことを、実際にVCCにアクセスして確認します。

In [ ]:
from common import logsetting
from vcpsdk.vcpsdk import VcpSDK
vcp = VcpSDK(vcc_access_token)

### UnitGroup名の指定

構築環境の UnitGroup名を指定します。

> 「010-パラメータ設定.ipynb」と同じUnitGroup名(`ugroup_name`)を指定することで、既に入力したパラメータを引き継ぐことができます。

VCノードを作成時に指定した値を確認するために `group_vars` ファイル名の一覧を表示します。

In [ ]:
!ls -1 group_vars/*.yml | sed -e 's/^group_vars\///' -e 's/\.yml//' | sort

UnitGroup名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'OpenHPC'

ugroup_name = 

#### チェック

指定されたUnitGroup名が適切であることを確認します。

group_vars にfeature, partition 設定が記録されていることを確認します。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)

# group_vars にslurm_features, slurm_partitions が定義されていることを確認する
if not {"slurm_features", "slurm_partitions"} <= set(gvars.keys()):
    raise RuntimeError("Feature, Partition設定が未実施です")

Ansibleで接続できることを確認します。

In [ ]:
!ansible {ugroup_name}_master -m ping

## Feature定義

新しいFeature（計算ノードグループ）の定義を行います。

このセクションは以下の3つの章から構成されています：

1. **計算ノードのSlurm設定**: SlurmのFeature名の指定
2. **計算ノードのリソース設定**: VCノードの物理リソース（プロバイダ、インスタンスタイプ、GPU等）
3. **計算ノードのネットワーク設定**: IPアドレス、ホスト名、NFSマウント

### 計算ノードのSlurm設定

追加する計算ノードグループのFeature名を指定します。

**命名ガイドライン**（`{hardware}-{environment}` 形式を推奨）:

- 英小文字、数字、ハイフンのみ使用可能（1–64文字）
- 既存のFeature名と重複不可

**命名例**:

- `gpu-a100-lab2`（A100 GPU、研究室2）
- `cpu-azure`（汎用CPU、Azure）
- `highmem-onpremises`（大メモリ、mdx/オンプレ）

既存のFeature名を確認します。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)
for f in gvars.get("slurm_features", {}).keys():
    print(f)

追加する計算ノードグループのFeature名を次のセルで指定してください。

In [ ]:
# (例)
# feature_name = 'gpu-a100-lab2'
# feature_name = 'cpu-azure'
# feature_name = 'highmem-onpremises'

feature_name = 

#### チェック

指定されたFeature名が適切かチェックします。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(globals(), ["feature_name"], ["gvars"], link=False)

### 計算ノードのリソース設定

計算ノードの物理的なリソース（CPU、メモリ、GPU等）に関するパラメータを指定します。

#### クラウドプロバイダ

VCノードを作成するプロバイダ名を指定します。

In [ ]:
# (例)
# compute_provider = 'aws'
# compute_provider = 'azure'
# compute_provider = 'oracle'
# compute_provider = 'onpremises'   ### mdxの場合

compute_provider = 

#### Flavor

計算ノードに割り当てるリソースに対応する `flavor` の値を指定します。

次のセルを実行すると、指定したプロバイダで利用可能な `flavor` の一覧が表示されます。

In [ ]:
vcp.df_flavors(compute_provider)

上に表示された表の `flavor` の欄の値から、計算ノードとして利用するVCノードの `flavor` を選んで次のセルで指定してください。

In [ ]:
# (例)
# compute_flavor = 'medium'
# compute_flavor = 'gpu'
# compute_flavor = 'default'    ### mdxの場合

compute_flavor = 

#### インスタンスタイプ（オプション）

`flavor`で事前に定義されている以外のインスタンスタイプを利用したい場合は指定してください。

In [ ]:
# (例)
# compute_instance_type = 'g5.xlarge'              # AWS NVIDIA A10G
# compute_instance_type = 'Standard_NC4as_T4_v3'   # Azure NVIDIA T4
# compute_instance_type = 'VM.GPU2.1'              # Oracle Cloud  NVIDIA P100

#### ルートボリュームサイズ（オプション）

計算ノードのルートボリュームサイズを変更する必要がある場合は指定してください。

In [ ]:
# (例)
# compute_root_size = 100

#### GPU設定

計算ノードでGPUを利用するかどうかを設定します。

In [ ]:
# (例)
# compute_use_gpu = False  # GPU を利用しない
# compute_use_gpu = True   # GPU を利用する

compute_use_gpu = False

GPUを利用する場合は、1ノードあたりのGPU数を指定してください。

In [ ]:
# (例)
# compute_gpus = 1

#### ノード数

初期構築時に起動する計算ノード数を指定します。

In [ ]:
# (例)
# compute_nodes = 4

compute_nodes = 

初回構築時の計算ノード数は上のセルで指定した`compute_nodes`の値となります。

> 構築後に「081-計算ノードの追加.ipynb」「082-計算ノードの削除.ipynb」を実行することにより計算ノード数を変更することができます。

#### チェック

この章で指定したパラメータが適切かどうかチェックします。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(
    globals(),
    [
        "compute_provider",
        "compute_flavor",
        "compute_root_size",
        "compute_use_gpu",
        "compute_gpus",
        "compute_nodes",
    ],
    ["vcp"],
)

### 計算ノードのネットワーク設定

計算ノードのIPアドレス、ホスト名、NFSマウントポイントを設定します。

#### 計算ノードのIPアドレス

計算ノードに割り当てるIPアドレスレンジを指定します。

**IPレンジの指定方法**:

IPアドレスレンジは以下の形式で指定します:
- 短縮形式: `"172.30.2.141-150"` （第4オクテットのみ指定）
- 完全形式: `"172.30.2.141-172.30.2.150"` （開始と終了の完全なIPアドレス）

**複数レンジの指定**:

非連続なIPアドレスを使用する場合は、リストで複数のレンジを指定できます:
```python
compute_ip_ranges = ["172.30.2.141-150", "172.30.2.161-170"]
```

この例では、141-150と161-170の20個のIPアドレスが使用可能になります。

**IPアドレス数と compute_nodes の関係**:

- IPレンジから計算される総IP数は、`compute_nodes` 以上である必要があります
- 初回構築時は `compute_nodes` 個のノードが起動しますが、将来IPレンジで指定したノード数まで増やせます

**注意事項**:

- 他のノード（マスターノード、既存Feature）と重複しないアドレスを指定してください

VCノードに割り当て可能なIPアドレスの範囲を確認します。

> mdx上での構築の場合には、以下のセルではアドレス範囲を取得できないため、[mdxでの静的IPアドレス設定方法](https://docs.mdx.jp/ja/main/faq.html#ip)に従って割り当て可能なIPアドレス範囲を確認し、範囲内かつ未使用のIPアドレス範囲からcompute_ip_rangesを設定してください。

In [ ]:
if compute_provider != "onpremises":
    print(vcp.get_vpn_catalog(compute_provider).get('private_network_ipmask'))

計算ノードに割り当てるIPアドレスレンジを指定します。

In [ ]:
# (例)
# compute_ip_ranges = ["172.30.2.141-150"]                      # 10個のIPアドレス (141-150)
# compute_ip_ranges = ["172.30.2.141-172.30.2.150"]             # 10個のIPアドレス (141-150)
# compute_ip_ranges = ["172.30.2.141-150", "172.30.2.161-170"]  # 20個のIPアドレス（非連続）
# compute_ip_ranges = [                                         # 30個のIPアドレス
#     "172.30.2.141-150",
#     "172.30.2.161-170",
#     "172.30.2.201-210",
# ]

compute_ip_ranges = [
    
]

#### ホスト名テンプレート

計算ノードのホスト名テンプレートを指定します。

**ホスト名テンプレートとは**:

ホスト名にはFeatureに応じたプレフィックスにノードの通番を付与する構成とします。例えばプレフィックスを `gpu` とした場合、ホスト名（非修飾ホスト名）は `gpu1`, `gpu2`, `gpu3`, ... のようになります。

通番の部分を動的に生成するので、ホスト名のテンプレートをPythonの[書式指定文字列](https://docs.python.org/ja/3.13/library/string.html#formatspec)により指定してください。

**例**:

- `"c{}"` → c1, c2, c3, ...
- `"c{:02}"` → c01, c02, c03, ...
- `"gpu{}"` → gpu1, gpu2, gpu3, ...
- `"gpu{:02}"` → gpu01, gpu02, gpu03, ...

**Featureとの関連**:

ホスト名のプレフィックスは、通常Feature名の性質に合わせて命名します:
- GPU Feature → `gpu{}`
- CPU Feature → `c{}`
- 高メモリFeature → `mem{}`

In [ ]:
# (例)
# compute_hostname_template = "c{}"       # c1, c2, c3 ...
# compute_hostname_template = "c{:02}"    # c01, c02, c03, ...
# compute_hostname_template = "gpu{:02}"  # gpu01, gpu02, gpu03, ...

compute_hostname_template = 

#### 追加NFSマウントポイント（オプション）

計算ノードに追加のNFSマウントポイントを設定したい場合は、この節でNFSエントリを指定してください。

> この設定はオプションです。追加のNFSマウントポイントが不要な場合は、この節をスキップしてください。

計算ノードには、デフォルトで以下の2つのNFSマウントポイントが設定されます：

* `/opt/ohpc/pub`
  - マスターノードの `/ohpc/pub` をマウント（読み取り専用）
* `/home`
  - マスターノードの `/home` をマウント（読み書き可能）

これらに加えて他のNFSサーバーからマウントする必要がある場合、以下のセルの`compute_optional_nfs_entries`でNFSエントリを追加することができます。

入力形式は`/etc/fstab`の書式に従います。fstabの記述例を以下に示します。参考にしてセルを設定して下さい。
```
server1.example.com:/data /mnt/nfs/data nfs nfsvers=4,rw,nodev 0 0
192.168.1.100:/shared /mnt/shared nfs nfsvers=4,rw 0 0
```

`compute_optional_nfs_entries`は複数行の文字列として指定してください。

In [ ]:
compute_optional_nfs_entries = """

"""

#### チェック

この章で指定したパラメータが適切かどうかチェックします。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
%run scripts/group.py
master_ipaddress = load_group_var(ugroup_name, "master_ipaddress")
check_parameters_with_keys(
    globals(),
    [
        "compute_ip_ranges",
        "compute_hostname_template",
        ("compute_optional_nfs_entries", None),
    ],
    [
        "vcp",
        "gvars",
        "compute_provider",
        "compute_flavor",
        "compute_nodes",
        "master_ipaddress"
    ],
)

### パラメータ保存

Feature定義に関するパラメータを`group_vars`ファイルに保存します。

In [ ]:
%run scripts/utils.py
%run scripts/group.py

feature_name, feature_config = generate_slurm_features_params(vars())
update_slurm_features(ugroup_name, feature_name, feature_config)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## Partition設定

新たに定義したFeatureを追加するPartitionの指定を行います。

### Partition名の指定

現在の登録状況を確認します。既に登録されているPartitionとそれに属するNodesetのリストを表示します。Nodeset名は`ns-{feature_name}`のようなルールでFeature名から導出された値になります。

In [ ]:
for partition in gvars["slurm_partitions"].keys():
    nodesets = gvars["slurm_partitions"][partition]["nodesets"]
    print(f"""  - {partition}:
      Nodesets: {', '.join(nodesets)}""")

Featureを追加するPartitionを指定してください。既存のPartition名を指定することも、新規のPartition名を指定することも、どちらも可能です。

In [ ]:
# Partition名の指定
# (例)
# partition_name = "gpu"      # 既存Partitionに追加する場合
# partition_name = "highmem"  # 新規Partitionを作成する場合

partition_name = 

指定されたPartition名が既存のものかどうかを確認します。

In [ ]:
add_to_existing = partition_name in gvars["slurm_partitions"].keys()
if add_to_existing:
    print(f"既存のPartition {partition_name} に {feature_name} のノードを追加します。")
else:
    print(f"{feature_name} のノードで構成される新たなPartition {partition_name} を登録します。")

### アクセス制御設定（オプション）

Partitionへのアクセスを特定のグループに制限したい場合は、この節で設定してください。この設定はオプションです。アクセス制限が不要な場合は、この節をスキップしてください。

**アクセス制御とは**:

SlurmのPartitionには、`AllowGroups`パラメータを使用してアクセスできるUNIXグループを制限できます。これにより、特定のユーザーグループのみが特定のPartitionにジョブを投入できるようになります。

**使用例**:

- GPU Partitionを特定の研究グループに制限
  ```python
  partition_allow_groups = ["gpu-users", "research-team"]
  ```
- 管理者のみがアクセス可能なテストPartition
  ```python
  partition_allow_groups = ["admin"]
  ```
- 全ユーザーがアクセス可能（デフォルト）
  ```python
  partition_allow_groups = []  # 空リストまたは未定義
  ```

**注意事項**:

- 指定するグループ名は、計算ノードおよびマスターノードのUNIXシステムに存在する必要があります
- グループ管理は別途Ansible等で実施する必要があります
- 空リスト `[]` または未定義の場合は、アクセス制限なし（全ユーザーがアクセス可能）


アクセス制限設定は新規パーティションを登録する場合のみ指定可能です。既存のPartitionのアクセス制御設定を変更する場合は「085-Partitionの変更.ipynb」を用いてください。

In [ ]:
# (例)
# partition_allow_groups = ["all-users"]           # all-usersグループのみ
# partition_allow_groups = ["gpu-users", "admin"]  # gpu-usersとadminグループ
# partition_allow_groups = []                      # 制限なし（全ユーザー）

partition_allow_groups = [
    
]

既存のパーティションを対象としている場合に誤って`partition_allow_groups`を設定してしまった場合は、コードセルで以下の内容を実行してください。

```python
del partition_allow_groups
```

### パラメータの保存

この章で指定したパラメータを`group_vars`ファイルに保存します。

値を保存する前に、指定された値の簡単なチェックを行います。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(
    globals(),
    ["partition_name", "partition_allow_groups"],
    ["gvars", "add_to_existing"],
)

パラメータを `group_vars` ファイルに保存します。

In [ ]:
%run scripts/utils.py
%run scripts/group.py
gvars = load_group_vars(ugroup_name)
nodeset_name = f"ns-{feature_name}"
if add_to_existing:
    partition_config = gvars["slurm_partitions"][partition_name]
    partition_config["nodesets"].append(nodeset_name)
else:
    partition_config = {"nodesets": [nodeset_name], "default": False}
    if "partition_allow_groups" in vars() and partition_allow_groups:
        partition_config["allow_groups"] = partition_allow_groups

update_slurm_partitions(ugroup_name, partition_name, partition_config)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## Slurmへの反映

構成定義の変更をSlurmの設定ファイルに反映し、その再読み込みを行います。

Ansibleを使用してマスターノードの以下のファイルを更新します：

- **slurm.conf**: Slurmの設定ファイル
  - Feature/Nodeset/Partition定義
  - MaxNodeCount（全Featureの合計ノード数）
  - GPU使用時はGresTypes設定
- **hosts.ohpc**: CoreDNS用のhostsファイル
  - 全計算ノードのホスト名とIPアドレスのマッピング

まずチェックモードで実行します。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v -CD playbooks/setup-slurm.yml

実際に変更を行います。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v playbooks/setup-slurm.yml

生成された設定を確認します：

1. **slurm.conf**: Slurm設定ファイルの内容確認
   - MaxNodeCount: 全Featureのノード数合計が正しいか
   - GresTypes: GPU使用時に設定されているか
   - NodeSet: `ns-{feature_name}` 形式で定義されているか
   - PartitionName: パーティション定義が正しいか

2. **hosts.ohpc**: CoreDNS用hostsファイルの内容確認
   - 全計算ノードのエントリが記載されているか
   - IPアドレスとホスト名のマッピングが正しいか

In [ ]:
# 1. slurm.conf の確認
print("=== Slurm Configuration ===")
!ansible {ugroup_name}_master \
    -m shell -a "scontrol show config | grep -E 'MaxNodeCount|GresTypes'"

print("\n=== Partition Configuration ===")
!ansible {ugroup_name}_master \
    -a "grep -wE 'PartitionName|NodeSet' /etc/slurm/slurm.conf"

# 2. hosts.ohpc の確認
print("\n=== Hosts File (hosts.ohpc) ===")
!ansible {ugroup_name}_master -a "cat /opt/ohpc/pub/etc/hosts.ohpc"

## mdx VMの起動

計算ノードとなるmdx VMを起動し、VCノードとして使用できる状態にします。mdx以外のクラウドを利用している場合は、この章をスキップして次章の「VCノード起動」から実行してください。

### mdx操作の準備

In [ ]:
from getpass import getpass
mdx_token = getpass("mdx API token")

IPv4接続を強制するように設定します。

In [ ]:
%run scripts/mdx_ops.py
use_ipv4_only()

mdxプラグインを読み込みます。

In [ ]:
from vcpsdk.plugins.mdx_ext import MdxResourceExt
mdx = MdxResourceExt(mdx_token)
mdx.set_current_project_by_name(gvars['mdx_project_name'])

### 計算ノード用VMパラメータの設定

Feature設定からVMスペックを生成します。

In [ ]:
%run scripts/mdx_ops.py
import os

with open(os.path.expanduser(gvars['ssh_public_key_path'])) as f:
    shared_key = f.read()

# Feature設定からVMスペックを生成
c_spec = mdx_get_compute_vm_spec_from_feature(
    mdx, feature_config, gvars['mdx_segment_id'], shared_key
)

In [ ]:
# service_level の設定
from vcpsdk.plugins.mdx_ext import MDX_VM_SPEC_SCHEMA

if not 'service_level' in MDX_VM_SPEC_SCHEMA['properties']:
    MDX_VM_SPEC_SCHEMA['properties'].update(
        {'service_level': {'type': 'string'}}
    )

c_spec.update({'service_level': 'guarantee'})

### VMデプロイ

計算ノードのVMをデプロイします。

In [ ]:
# デプロイ対象のホスト名リストを算出
hostname_template = feature_config["hostname_template"]
compute_hostnames = [
    hostname_template.format(i + 1)
    for i in range(feature_config["nodes"])
]
compute_ip_addresses = feature_config["ip_addresses"][:feature_config["nodes"]]

In [ ]:
# VM初期パスワード
initial_passwd = 'openhpcv3_mdx_vm_initial_password'

In [ ]:
!pip install --no-deps jupyter-autotime
%load_ext autotime

In [ ]:
%run scripts/mdx_ops.py

mdx_deploy_vms(mdx, compute_hostnames, c_spec,
               project=gvars['mdx_project_name'], verbose=True)

### IPアドレスの変更

In [ ]:
%run scripts/mdx_ops.py

# {新IPアドレス: ホスト名} の辞書
add_etc_hosts = dict(zip(compute_ip_addresses, compute_hostnames))

# 初期パスワード設定
mdx_set_init_passwd(mdx, compute_hostnames,
                    gvars['ssh_private_key_path'],
                    initial_passwd)

# IPアドレス変更
mdx_change_ipaddrs(mdx, add_etc_hosts,
                   os.path.expanduser(gvars['ssh_private_key_path']),
                   verbose=True)

### VCノード向け設定

mdx VMをVCノードとして使用できるように設定します。

In [ ]:
%run scripts/mdx_ops.py

vcppubkey = vcp.get_publickey()
mdx_init_vcp(list(add_etc_hosts.keys()),
             os.path.expanduser(gvars['ssh_private_key_path']),
             vcppubkey)

## VCノード起動

新たに定義したFeatureの計算ノードグループを起動します。

### spec を指定する

Featureとして指定したパラメータを`spec` に設定します。

In [ ]:
# Feature設定を取得
feature_config = gvars['slurm_features'][feature_name]

# VCP spec の設定
spec_compute = vcp.get_spec(feature_config['provider'], feature_config.get('flavor'))

# コンテナイメージの設定
img_tag = "compute-gpu-3.4-multi" if feature_config["use_gpu"] else "compute-3.4-multi"
spec_compute.image = f"harbor.vcloud.nii.ac.jp/vcp/openhpc:{img_tag}"

# GPU設定
if feature_config['use_gpu']:
    spec_compute.gpus = 'all'
    if feature_config['provider'] != 'onpremises':
        spec_compute.cloud_image = 'niivcp-25.04.0-x86_64-ubuntu24.04-gpu-a-r1.0'
    
# IP アドレスとノード数の設定
if feature_config['provider'] == 'onpremises':
    spec_compute.num_nodes = feature_config['nodes']
spec_compute.ip_addresses = feature_config['ip_addresses'][:feature_config['nodes']]

# インスタンスタイプとディスクサイズ
if 'instance_type' in feature_config and feature_config['instance_type']:
    # provider別の処理
    match feature_config['provider']:
        case 'aws':
            spec_compute.instance_type = feature_config['instance_type']
            if 'root_size' in feature_config:
                spec_compute.volume_size = feature_config['root_size']
        case 'azure':
            spec_compute.vm_size = feature_config['instance_type']
            if 'root_size' in feature_config:
                spec_compute.disk_size = feature_config['root_size']
        case 'oracle':
            spec_compute.shape = feature_config['instance_type']
            if 'root_size' in feature_config:
                spec_compute.boot_volume_size_in_gbs = feature_config['root_size']

# SSH公開鍵の設定
spec_compute.set_ssh_pubkey(gvars["ssh_public_key_path"])

# mdx
if 'mdx_ssh_user_name' in gvars:
    spec_compute.user_name = gvars['mdx_ssh_user_name']

# DNS設定
spec_compute.dns = [gvars["master_ipaddress"]]
spec_compute.dns_search = [gvars["dns_domain"]]
spec_compute.add_host = [f"master:{gvars['master_ipaddress']}"]

# bind mount設定
spec_compute.params_v = [
    '/sys/fs/cgroup:/sys/fs/cgroup:rw',
    '/opt/var/lib/docker:/var/lib/docker',
    '/lib/modules:/lib/modules:ro',
]

ここまで `spec` に設定した内容を表示してみます。

In [ ]:
print(spec_compute)

ベースコンテナに指定する環境変数を設定します。

In [ ]:
# 環境変数の設定
munge_key = spec_env_munge_key(gvars, vcp, vcc_access_token, verify=gvars.get("verify_ssl_certificate", True))
spec_compute.params_e.extend([
    f'MUNGE_KEY={munge_key}',
    f'MASTER_HOSTNAME={gvars["master_hostname"]}',
    f'CLUSTER={gvars["ugroup_name"]}',
    f'FEATURE={feature_name}',
    'ENABLE_NSS_SLURM=1',
])

if feature_config['use_gpu'] and feature_config['gpus'] > 0:
    spec_compute.params_e.append(f'GPUS={feature_config["gpus"]}')

if feature_config.get('nfs_entries'):
    spec_compute.params_e.append(
        f"OPTIONAL_NFS_FSTAB_BASE64={feature_config['nfs_entries']}"
    )

ここまで `spec` に設定した内容を表示してみます。

In [ ]:
print(spec_compute)

### VCユニットの作成

新たなVCユニットを作成し、計算ノードを起動します。

In [ ]:
# UnitGroupの取得
ug = vcp.get_ugroup(ugroup_name)

# Unit作成（VCユニット名を使用）
vc_unit_name = feature_name
unit_compute = ug.create_unit(vc_unit_name, spec_compute)

VCノードの状態が `RUNNING` になっていることを確認します。

In [ ]:
unit_compute = ug.get_unit(vc_unit_name)
if any([node.state != 'RUNNING' for node in unit_compute.find_nodes()]):
    raise RuntimeError('ERROR: not running')

起動したVCノードの一覧を表示します。

In [ ]:
unit_compute.df_nodes()

### Ansibleのセットアップ

起動した計算ノードに対してAnsibleで操作を行うための設定を行います。

#### inventory.yml の更新

新しい計算ノードをAnsibleインベントリに追加します。

In [ ]:
%run scripts/group.py

# 既存のinventoryを読み込む
try:
    with open('inventory.yml', 'r') as f:
        import yaml
        inventory = yaml.safe_load(f)
except FileNotFoundError:
    inventory = {}

# 新しい計算ノードのグループ名
compute_group = f"{ugroup_name}_{vc_unit_name}"

# 新しい計算ノードを追加
inventory["all"]["children"][ugroup_name]["children"][compute_group] = {
    "hosts": {addr: {} for addr in unit_compute.find_ip_addresses()},
}

# inventory を更新
update_inventory_yml(inventory)

# inventoryのノード構成を確認
!ansible-inventory --graph {ugroup_name}

#### known_hostsの更新

`~/.ssh/known_hosts` の更新を行います。

In [ ]:
from time import sleep

for retry in range(10):
    try:
        for addr in unit_compute.find_ip_addresses():
            !ssh-keygen -R {addr} 2> /dev/null
            !ssh-keyscan -H {addr} >> ~/.ssh/known_hosts 2> /dev/null
        print("✓ known_hostsを更新しました")
        break
    except:
        if retry < 9:
            print(f"リトライ中... ({retry + 1}/10)")
            sleep(10)
        else:
            raise

#### 疎通確認

新たに起動した計算ノードへの疎通確認を行います。

In [ ]:
!ansible {compute_group} -m ping

管理者権限によるコマンド実行が可能かどうかを確認します。

In [ ]:
!ansible {compute_group} -b -a 'whoami'

### Slurmノード登録確認

計算ノードを実際に起動するまでは、まだNodesetに参加していない状態のため新たなPartitionの設定を認識しない場合があります。計算ノード起動後に scontrol reconfigure を実行してSlurmの状態を更新します。

In [ ]:
!ansible {ugroup_name}_master -b \
    -a "scontrol reconfigure"

Slurmクラスタのノードの状態を確認します。

In [ ]:
!ansible {ugroup_name}_master \
    -a "sinfo -o '%P %N %F %f'"